- https://www.youtube.com/watch?v=3nM532iNf2Y

In [1]:
from IPython.display import Image

In [3]:
Image(url='https://arxiv.org/html/2409.19256v1/x3.png', width=400)

- dataflow graph
    - machineA/machineB: computation with dependencies executes sequencially
    - Actor training / critic training：models in the same stage and on different GPUs can be parallelized
    - ref/rm: colocate, sequentially

### EntryPoint

- main_ppo.TaskRunner.run
    - resource_pool_manager

```python
global_pool_id = "global_pool"
resource_pool_spec = {
    global_pool_id: [config.trainer.n_gpus_per_node] * config.trainer.nnodes,
}

self.mapping[Role.ActorRollout] = global_pool_id
self.mapping[Role.Critic] = global_pool_id
self.mapping[Role.RewardModel] = "global_pool"
self.mapping[Role.RefPolicy] = "global_pool"

resource_pool_manager = ResourcePoolManager(resource_pool_spec=resource_pool_spec, mapping=self.mapping)

self.role_worker_mapping[Role.ActorRollout] = ray.remote(actor_rollout_cls)
self.role_worker_mapping[Role.Critic] = ray.remote(CriticWorker)
self.role_worker_mapping[Role.RewardModel] = ray.remote(RewardModelWorker)
self.role_worker_mapping[Role.RefPolicy] = ray.remote(ref_policy_cls)

trainer = RayPPOTrainer(config, 
                        role_worker_mapping=self.role_worker_mapping, 
                        resource_pool_manager=resource_pool_manager, ...)

trainer.init_workers()

trainer.fit()
```

#### `trainer.init_workers()`

- hybrid engine
    - 将传统上分离的 Actor（策略网络）和 Rollout（推理生成）功能融合到同一个工作器中，即同一GPU既可训练也可推理

- 每个 wg（worker group）对应一个 `resource_pool`（也就是若干个 gpu 资源）

```python
for resource_pool, class_dict in self.resource_pool_to_cls.items():
    worker_dict_cls = create_colocated_worker_cls(class_dict=class_dict)
    wg_dict = self.ray_worker_group_cls(
        resource_pool=resource_pool,
        ray_cls_with_init=worker_dict_cls,
        **wg_kwargs,
    )
    spawn_wg = wg_dict.spawn(prefix_set=class_dict.keys())
    all_wg.update(spawn_wg)
```
 
```python
resource_pool_to_cls = {
    global_pool: {
        "actor_rollout": RayClassWithInitArgs(ActorRolloutRefWorker, config=actor_config, role="actor_rollout"),
        "critic": RayClassWithInitArgs(CriticWorker, config=critic_config),
        "ref": RayClassWithInitArgs(RefPolicyWorker, config=ref_config, role="ref"),
    },
    reward_pool: {  # 如果启用独立奖励模型池
        "rm": RayClassWithInitArgs(RewardModelWorker, config=rm_config),
    }
}
worker_dict_cls = create_colocated_worker_cls(class_dict={
    "actor_rollout": ActorRolloutRefWorker,
    "critic": CriticWorker, 
    "ref": RefPolicyWorker,
})
```

### single-controller vs. multi-controller

In [4]:
Image(url='./figs/verl-single-controller.png', width=400)

- 不断地更新（维护）data（DataProto）
- Between worker procedures, verl adopts a single-controller paradigm to maximize the flexibility, which allows the users to
    - focus on the dataflow graph
    - without worrying about the distributed implementation.
- verl runs the worker procedures **sequentially** within the global resource pool by default.

- multri-controller (inside a worker procedure)
    - spmd：（Single Program, Multiple Data） = 所有 GPU/进程同时跑同一段程序，只是各自处理不同的数据分片。
        - run the same program
        - but process different data based on the distributed env variables like RANK
    - SPMD is the programming model of most popular distributed methods
        - DP (DDP, ZeRO, FSDP)
        - TP, PP, SP

### fsdp workers

- @register(dispatch_mode=Dispatch.DP_COMPUTE_PROTO) 的 `generate_sequences`：数据按 DP 维度切分，每个 rank 用 vLLM 做自回归生成，随后由 actor 端重算 logprob；
- @register(dispatch_mode=Dispatch.ONE_TO_ALL) 的 `init_model`：广播；